# 分布式训练


## 分布式计算
- 数据放在分布式文件系统上
- 多台机器，每台机器有多个 GPU
- 多个参数服务器

跨网络交换数据，本质和单机多卡并行没区别，只是通信开销更大。


## GPU 机器架构
```mermaid
graph LR
    S[交换机] ---|"1.25 GB/s <br> (10 Gbit Ethernet)"|C[CPU]
    C ---|"15.75 GB/s <br> (PCIe 3.0 x16)"|P[PCIe 交换]
    P ---|"63 GB/s"|G1[GPU 1]
    P ---|"63 GB/s"|G2[GPU 2]
    P ---|"63 GB/s"|G3[GPU 3]
    P ---|"63 GB/s"|G4[GPU 4]
```
- GPU 之间通信很快
- CPU 和 GPU 之间通信稍慢
- 跨设备通信很慢

所以尽量减少跨设备通信，可以多用内存在本地通信。


## 计算一个小批量
```mermaid
sequenceDiagram
    participant DS as 数据服务器
    participant PS as 参数服务器
    participant CS as 计算服务器
    participant GPU as GPU

    DS->>CS: 读取小批量数据
    CS->>GPU: 将数据切分到每个 GPU
    PS->>CS: 获取模型参数
    CS->>GPU: 将参数复制到每个 GPU 上
    GPU->>GPU: 每个 GPU 计算梯度
    GPU->>CS: 所有 GPU 上的梯度求和
    CS->>PS: 梯度传回参数服务器
    PS->>PS: 参数服务器对梯度求和并更新参数
```


## 性能
### 同步 SGD
每个计算服务器同步计算一个批量，称为同步 SGD。
- $n$ 个 GPU，每个 GPU 计算 $b$ 个样本
- 同步 SGD 等价于单 GPU 运行批量大小为 $nb$ 的 SGD
- 理想情况，$n$ 个 GPU 训练速度是单 GPU 的 $n$ 倍

### 通信与计算
- 单 GPU 上计算 $b$ 个样本的时间 $=t_c$
- 每个计算服务器发送和接收参数的时间 $=t_w$
- 每个批量计算时间 $=max(t_c, t_w)$
  - 选取足够大的 $b$ 使得 $t_c > t_w$，减少等待时间。
  - 但更大的批量大小 $nb$ 可能导致收敛变慢